In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os

import duckdb
import polars as pl

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (base for base in (CURRENT, CURRENT.parent) if (base / 'data' / 'raw' / 'qna.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'File data/raw/qna.csv tidak ditemukan. Jalankan notebook dari root proyek atau '
        'notebooks/ dan pastikan file tersedia di data/raw/qna.csv.'
    )

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'qna.csv'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED / 'qna_clean.parquet'
REPORT_PATH = PROCESSED / 'quality_report.json'

EXPECTED_COLUMNS = [
    'id', 'title', 'user', 'question', 'question_clean',
    'question_post_date', 'answer_by', 'answer_post_date',
    'answer', 'answer_clean',
]
REQUIRED_COLUMNS = ('id', 'question', 'answer')
DATE_COLUMNS = ('question_post_date', 'answer_post_date')

print('Project root :', PROJECT_ROOT)
print('File sumber  :', DATA_PATH)
print(f'Ukuran       : {DATA_PATH.stat().st_size / (1024**3):.2f} GiB')
print('Polars       :', pl.__version__)
print('DuckDB       :', duckdb.__version__)

Project root : /home/jovyan/work
File sumber  : /home/jovyan/work/data/raw/qna.csv
Ukuran       : 2.64 GiB
Polars       : 2.0.0
DuckDB       : 2.0.0


In [2]:
raw = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=False,
    try_parse_dates=False,
)
actual_schema = raw.collect_schema()
missing_columns = set(EXPECTED_COLUMNS) - set(actual_schema.names())
if missing_columns:
    raise ValueError(f'Kolom wajib dari skema dataset tidak ditemukan: {sorted(missing_columns)}')

# Milestone 1 mendapati semua kolom bertipe String.
non_string_cols = [(k, v) for k, v in actual_schema.items() if k in EXPECTED_COLUMNS and v != pl.String]
if non_string_cols:
    raise TypeError(f'Kolom sumber diharapkan bertipe String, ditemukan: {non_string_cols}')

raw = raw.select(EXPECTED_COLUMNS)
n_before = raw.select(pl.len()).collect(engine='streaming').item()
print('Jumlah baris awal:', f'{n_before:,}')
print('Kolom:', ', '.join(raw.collect_schema().names()))

Jumlah baris awal: 681,203
Kolom: id, title, user, question, question_clean, question_post_date, answer_by, answer_post_date, answer, answer_clean


In [3]:
con = duckdb.connect(database=':memory:')
# SUMMARIZE memberikan column_name, column_type, min, max, approx_unique, null_percentage, dll.
result = con.execute(
    'SUMMARIZE SELECT * FROM read_csv(?, header = true, all_varchar = true, sample_size = 20000)',
    [str(DATA_PATH)],
)
profil_duckdb = pl.DataFrame(
    result.fetchall(),
    schema=[column[0] for column in result.description],
    orient='row',
    infer_schema_length=None,
)
profil_duckdb.write_csv(PROCESSED / 'duckdb_summarize.csv')
print('Profil DuckDB disimpan ke data/processed/duckdb_summarize.csv')
profil_duckdb

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Profil DuckDB disimpan ke data/processed/duckdb_summarize.csv


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""id""","""VARCHAR""","""576affb8abcdce1d7101605f""","""65e141177d39360025de88ba""",429097,null,null,null,null,null,681203,0.00
"""title""","""VARCHAR""",""" Penyebab ruam atau bintik me…","""zat yang terkandung pada yogur…",388208,null,null,null,null,null,681203,0.00
"""user""","""VARCHAR""","""'*a""","""😘*""",46438,null,null,null,null,null,681203,0.00
"""question""","""VARCHAR""",""" hai dok,sa…","""👋 dok nama sy rizka, usia 20 t…",405214,null,null,null,null,null,681203,0.00
"""question_clean""","""VARCHAR""","""!!Malam dok. istri saya serin…","""🙏🏻Tim dokter, sharing dong, ta…",443905,null,null,null,null,null,681203,0.01
"""question_post_date""","""VARCHAR""","""Fri Apr 01 2016 08:05:00 GMT+0…","""Wed Sep 30 2020 23:52:00 GMT+0…",399257,null,null,null,null,null,681203,0.00
"""answer_by""","""VARCHAR""","""'A**a""","""ﻳﻮ****ﺭ""",4260,null,null,null,null,null,681203,0.00
"""answer_post_date""","""VARCHAR""","""Fri Apr 01 2016 00:03:00 GMT+0…","""Wed Sep 30 2020 23:57:00 GMT+0…",381656,null,null,null,null,null,681203,0.00
"""answer""","""VARCHAR""",""" Kevin yang baik, …","""hello Terima kasih sudah berta…",414379,null,null,null,null,null,681203,0.00


In [4]:
METADATA_COLUMNS = ('id', 'title', 'user', 'answer_by')

def normalize_missing(lf: pl.LazyFrame) -> pl.LazyFrame:
    edits = []
    for name in EXPECTED_COLUMNS:
        val = pl.col(name)
        trimmed = val.str.strip_chars()
        if name in METADATA_COLUMNS:
            edits.append(pl.when(trimmed == '').then(None).otherwise(trimmed).alias(name))
        else:
            edits.append(pl.when(trimmed == '').then(None).otherwise(val).alias(name))
    return lf.with_columns(edits)

normalized = normalize_missing(raw)

def null_table(lf: pl.LazyFrame, stage: str) -> pl.DataFrame:
    n = lf.select(pl.len()).collect(engine='streaming').item()
    counts = lf.select([pl.col(c).null_count().alias(c) for c in EXPECTED_COLUMNS]).collect(engine='streaming')
    values = counts.row(0, named=True)
    return pl.DataFrame({
        'tahap': [stage] * len(EXPECTED_COLUMNS),
        'kolom': EXPECTED_COLUMNS,
        'jumlah_null': [int(values[c]) for c in EXPECTED_COLUMNS],
        'persen_null': [round(100 * int(values[c]) / n, 4) if n else 0 for c in EXPECTED_COLUMNS],
    }).sort('jumlah_null', descending=True)

null_before = null_table(normalized, 'sebelum_cleaning')
null_before.write_csv(PROCESSED / 'missing_before.csv')
null_before

tahap,kolom,jumlah_null,persen_null
str,str,i64,f64
"""sebelum_cleaning""","""question_clean""",97,0.0142
"""sebelum_cleaning""","""question""",43,0.0063
"""sebelum_cleaning""","""answer_clean""",15,0.0022
"""sebelum_cleaning""","""title""",9,0.0013
"""sebelum_cleaning""","""user""",4,0.0006
"""sebelum_cleaning""","""answer""",2,0.0003
"""sebelum_cleaning""","""id""",0,0.0
"""sebelum_cleaning""","""question_post_date""",0,0.0
"""sebelum_cleaning""","""answer_by""",0,0.0


In [5]:
required_ok = pl.all_horizontal([pl.col(c).is_not_null() for c in REQUIRED_COLUMNS])
valid_rows = normalized.filter(required_ok)
n_valid = valid_rows.select(pl.len()).collect(engine='streaming').item()
n_rejected = n_before - n_valid

print(f'Baris awal                 : {n_before:,}')
print(f'Baris valid (3 kolom wajib): {n_valid:,}')
print(f'Baris ditolak              : {n_rejected:,}')
print('Catatan: baris ditolak tidak mengubah CSV sumber.')

Baris awal                 : 681,203
Baris valid (3 kolom wajib): 681,158
Baris ditolak              : 45
Catatan: baris ditolak tidak mengubah CSV sumber.


In [6]:
unique_rows = valid_rows.unique(subset=EXPECTED_COLUMNS, keep='any', maintain_order=False)
print('Aturan deduplikasi: persamaan penuh seluruh 10 kolom, bukan ID saja.')

Aturan deduplikasi: persamaan penuh seluruh 10 kolom, bukan ID saja.


In [7]:
DATE_PATTERN = r'^([A-Za-z]{3}\s+[A-Za-z]{3}\s+\d{1,2}\s+\d{4}\s+\d{2}:\d{2}:\d{2}\s+GMT[+-]\d{4})'

def parsed_date(name: str) -> pl.Expr:
    return (
        pl.col(name)
        .str.extract(DATE_PATTERN, 1)
        .str.replace(r'\s+GMT', ' ')
        .str.strptime(
            pl.Datetime(time_zone='UTC'),
            format='%a %b %d %Y %H:%M:%S %z',
            strict=False,
        )
        .alias(f'{name}_utc')
    )

def add_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        parsed_date('question_post_date'),
        parsed_date('answer_post_date'),
        pl.col('question').str.len_chars().alias('question_length'),
        pl.col('answer').str.len_chars().alias('answer_length'),
    )

with_features = add_features(unique_rows)
print('Fitur baru: 2 tanggal UTC dan 2 panjang teks.')

Fitur baru: 2 tanggal UTC dan 2 panjang teks.


In [8]:
length_stats = with_features.select([
    pl.col(c).quantile(q).alias(f'{c}_{label}')
    for c in ('question_length', 'answer_length')
    for q, label in ((0.25, 'q1'), (0.75, 'q3'))
]).collect(engine='streaming').row(0, named=True)

limits = {}
for c in ('question_length', 'answer_length'):
    q1, q3 = length_stats[f'{c}_q1'], length_stats[f'{c}_q3']
    if q1 is None or q3 is None:
        raise ValueError(f'Kolom {c} tidak memiliki cukup data untuk IQR')
    iqr = q3 - q1
    limits[c] = {
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'batas_bawah': max(0.0, q1 - 1.5 * iqr),
        'batas_atas': q3 + 1.5 * iqr,
    }

outlier_table = pl.DataFrame([{'kolom': c, **v} for c, v in limits.items()])
outlier_table.write_csv(PROCESSED / 'outlier_iqr.csv')
outlier_table

kolom,q1,q3,iqr,batas_bawah,batas_atas
str,f64,f64,f64,f64,f64
"""question_length""",197.0,437.0,240.0,0.0,797.0
"""answer_length""",1275.0,2029.0,754.0,144.0,3160.0


In [9]:
prepared = with_features.with_columns(
    (pl.col('question_length') > limits['question_length']['batas_atas']).alias('is_outlier_question_length'),
    (pl.col('answer_length') > limits['answer_length']['batas_atas']).alias('is_outlier_answer_length'),
).with_columns(
    (
        pl.col('question_post_date_utc').is_not_null()
        & pl.col('answer_post_date_utc').is_not_null()
        & (pl.col('answer_post_date_utc') < pl.col('question_post_date_utc'))
    ).alias('is_answer_before_question'),
)
print('Nilai outlier dan anomali waktu hanya ditandai, tidak mengubah teks sumber.')

Nilai outlier dan anomali waktu hanya ditandai, tidak mengubah teks sumber.


In [10]:
from copy import deepcopy

fixture = {
    'id': 'ID-A', 'title': 'Topik', 'user': 'u1', 'question': '<p>Pertanyaan?</p>',
    'question_clean': 'Pertanyaan?',
    'question_post_date': 'Fri Jul 14 2023 13:25:00 GMT+0000 (Coordinated Universal Time)',
    'answer_by': 'Dokter',
    'answer_post_date': 'Fri Jul 14 2023 17:59:00 GMT+0000 (Coordinated Universal Time)',
    'answer': '<p>Jawaban 1</p>', 'answer_clean': 'Jawaban 1',
}
fixture_b = deepcopy(fixture)
fixture_b['answer'] = '<p>Jawaban 2</p>'
fixture_b['answer_clean'] = 'Jawaban 2'
fixture_invalid = deepcopy(fixture)
fixture_invalid['id'] = 'ID-B'
fixture_invalid['question'] = '   '
fixture_test = pl.DataFrame([fixture, fixture_b, fixture, fixture_invalid]).lazy()
fixture_valid = normalize_missing(fixture_test).filter(required_ok)
fixture_unique = fixture_valid.unique(subset=EXPECTED_COLUMNS, keep='any', maintain_order=False)
fixture_out = add_features(fixture_unique).collect()
assert fixture_out.height == 2, 'Deduplikasi atau filter wajib keliru'
assert fixture_out['id'].n_unique() == 1, 'ID berulang harus diizinkan'
assert fixture_out['question_post_date_utc'].null_count() == 0, 'Format tanggal pada contoh gagal diparse'
print('Uji mini LULUS: 2 jawaban berbeda dipertahankan, 1 duplikat penuh dan 1 baris invalid dibuang.')

Uji mini LULUS: 2 jawaban berbeda dipertahankan, 1 duplikat penuh dan 1 baris invalid dibuang.


In [11]:
TMP_OUTPUT_PATH = PROCESSED / 'qna_clean.tmp.parquet'
if TMP_OUTPUT_PATH.exists():
    TMP_OUTPUT_PATH.unlink()

prepared.sink_parquet(str(TMP_OUTPUT_PATH), compression='zstd')
os.replace(TMP_OUTPUT_PATH, OUTPUT_PATH)

cleaned = pl.scan_parquet(OUTPUT_PATH)
n_after = cleaned.select(pl.len()).collect(engine='streaming').item()
n_exact_removed = n_valid - n_after

assert n_after <= n_valid <= n_before, 'Jumlah baris tidak konsisten'
assert n_after > 0, 'Output kosong; periksa sumber atau aturan cleaning'
print('Hasil tersimpan:', OUTPUT_PATH)
print(f'Baris awal: {n_before:,} | baris ditolak: {n_rejected:,} | duplikat penuh terhapus: {n_exact_removed:,} | akhir: {n_after:,}')

Hasil tersimpan: /home/jovyan/work/data/processed/qna_clean.parquet
Baris awal: 681,203 | baris ditolak: 45 | duplikat penuh terhapus: 266,969 | akhir: 414,189


In [12]:
validation = cleaned.select([
    pl.len().alias('baris_akhir'),
    *[pl.col(c).null_count().alias(f'null_{c}') for c in REQUIRED_COLUMNS],
    (pl.col('question_length') <= 0).sum().alias('panjang_pertanyaan_tidak_valid'),
    (pl.col('answer_length') <= 0).sum().alias('panjang_jawaban_tidak_valid'),
    (pl.col('question_post_date').is_not_null() & pl.col('question_post_date_utc').is_null()).sum().alias('tanggal_pertanyaan_gagal_parse'),
    (pl.col('answer_post_date').is_not_null() & pl.col('answer_post_date_utc').is_null()).sum().alias('tanggal_jawaban_gagal_parse'),
    pl.col('is_outlier_question_length').sum().alias('outlier_pertanyaan'),
    pl.col('is_outlier_answer_length').sum().alias('outlier_jawaban'),
    pl.col('is_answer_before_question').sum().alias('jawaban_lebih_awal'),
    pl.col('question_post_date_utc').is_not_null().sum().alias('tanggal_pertanyaan_terparse'),
    pl.col('answer_post_date_utc').is_not_null().sum().alias('tanggal_jawaban_terparse'),
]).collect(engine='streaming')
checks = validation.row(0, named=True)
assert int(checks['baris_akhir']) == n_after
assert all(checks[f'null_{c}'] == 0 for c in REQUIRED_COLUMNS)
assert checks['panjang_pertanyaan_tidak_valid'] == 0
assert checks['panjang_jawaban_tidak_valid'] == 0
assert n_before == n_rejected + n_exact_removed + n_after

# Parsing gagal total adalah sinyal format/tz sumber berubah; harus diperiksa.
for date_col, name in (("question_post_date", "tanggal_pertanyaan_terparse"), ("answer_post_date", "tanggal_jawaban_terparse")):
    if checks[name] == 0:
        print(f'PERINGATAN: tidak ada nilai {date_col} yang berhasil diparse. Periksa format tanggal!')

print('VALIDASI DASAR LULUS (ID, pertanyaan, jawaban dan panjang teks).')
validation

VALIDASI DASAR LULUS (ID, pertanyaan, jawaban dan panjang teks).


baris_akhir,null_id,null_question,null_answer,panjang_pertanyaan_tidak_valid,panjang_jawaban_tidak_valid,tanggal_pertanyaan_gagal_parse,tanggal_jawaban_gagal_parse,outlier_pertanyaan,outlier_jawaban,jawaban_lebih_awal,tanggal_pertanyaan_terparse,tanggal_jawaban_terparse
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
414189,0,0,0,0,0,0,0,22975,13212,891,414189,414189


In [13]:
null_after = null_table(cleaned.select(EXPECTED_COLUMNS), 'setelah_cleaning')
null_after.write_csv(PROCESSED / 'missing_after.csv')

id_summary = (
    cleaned.group_by('id').len()
    .select(
        pl.len().alias('jumlah_id_unik'),
        (pl.col('len') > 1).sum().alias('jumlah_id_berulang'),
        pl.col('len').max().alias('maks_jawaban_per_id'),
    )
    .collect(engine='streaming')
    .row(0, named=True)
)
print('Ringkasan ID:', id_summary)
print('Missing setelah cleaning (kolom opsional boleh null):')
null_after

Ringkasan ID: {'jumlah_id_unik': 412207, 'jumlah_id_berulang': 1982, 'maks_jawaban_per_id': 2}
Missing setelah cleaning (kolom opsional boleh null):


tahap,kolom,jumlah_null,persen_null
str,str,i64,f64
"""setelah_cleaning""","""question_clean""",38,0.0092
"""setelah_cleaning""","""answer_clean""",12,0.0029
"""setelah_cleaning""","""user""",4,0.001
"""setelah_cleaning""","""title""",3,0.0007
"""setelah_cleaning""","""id""",0,0.0
"""setelah_cleaning""","""question""",0,0.0
"""setelah_cleaning""","""question_post_date""",0,0.0
"""setelah_cleaning""","""answer_by""",0,0.0
"""setelah_cleaning""","""answer_post_date""",0,0.0


In [14]:
def json_primitive(v):
    if isinstance(v, (int, float, str, bool)) or v is None:
        return v
    return str(v)

report = {
    'dataset': 'Indonesia Medical Q&A',
    'sumber': 'https://huggingface.co/datasets/abid/indonesia-medical-qna',
    'waktu_proses_utc': datetime.now(timezone.utc).isoformat(),
    'file_sumber': str(DATA_PATH.relative_to(PROJECT_ROOT)),
    'file_hasil': str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    'ukuran_sumber_bytes': DATA_PATH.stat().st_size,
    'jumlah_baris': {
        'awal': n_before,
        'ditolak_karena_wajib_kosong': n_rejected,
        'duplikat_penuh_dihapus': n_exact_removed,
        'akhir': n_after,
    },
    'kebijakan_missing': {
        'wajib': list(REQUIRED_COLUMNS),
        'wajib_kosong': 'baris dikeluarkan dari output, file mentah tetap disimpan',
        'opsional_kosong': 'dipertahankan NULL, tanpa imputasi fiktif',
        'whitespace_saja': 'dianggap NULL',
    },
    'kebijakan_deduplikasi': 'hanya identik pada 10 kolom sumber; ID berulang dipertahankan',
    'kebijakan_outlier': 'IQR panjang teks, flag saja (tanpa membuang/memotong teks)',
    'batas_iqr': limits,
    'hasil_validasi': {k: json_primitive(v) for k, v in checks.items()},
    'ringkasan_id': {k: json_primitive(v) for k, v in id_summary.items()},
    'missing_sebelum': null_before.to_dicts(),
    'missing_sesudah': null_after.to_dicts(),
    'catatan': [
        'Data berupa teks konsultasi medis; jangan unggah data baris mentah ke repo publik.',
        'Panjang teks yang ekstrem tidak otomatis kesalahan.',
        'Format waktu dikonversi ke UTC menggunakan offset GMT asli.',
        'Timestamp yang gagal diparse tetap ada di kolom teks asli dan dilaporkan.',
        'Status lisensi dataset perlu diverifikasi pada pemilik sumber.',
    ],
}
with open(REPORT_PATH, 'w', encoding='utf-8') as fp:
    json.dump(report, fp, ensure_ascii=False, indent=2, default=json_primitive)
print('Laporan tersimpan:', REPORT_PATH)
print(json.dumps({
    'baris': report['jumlah_baris'],
    'validasi': report['hasil_validasi'],
    'id': report['ringkasan_id'],
}, ensure_ascii=False, indent=2, default=json_primitive))

Laporan tersimpan: /home/jovyan/work/data/processed/quality_report.json
{
  "baris": {
    "awal": 681203,
    "ditolak_karena_wajib_kosong": 45,
    "duplikat_penuh_dihapus": 266969,
    "akhir": 414189
  },
  "validasi": {
    "baris_akhir": 414189,
    "null_id": 0,
    "null_question": 0,
    "null_answer": 0,
    "panjang_pertanyaan_tidak_valid": 0,
    "panjang_jawaban_tidak_valid": 0,
    "tanggal_pertanyaan_gagal_parse": 0,
    "tanggal_jawaban_gagal_parse": 0,
    "outlier_pertanyaan": 22975,
    "outlier_jawaban": 13212,
    "jawaban_lebih_awal": 891,
    "tanggal_pertanyaan_terparse": 414189,
    "tanggal_jawaban_terparse": 414189
  },
  "id": {
    "jumlah_id_unik": 412207,
    "jumlah_id_berulang": 1982,
    "maks_jawaban_per_id": 2
  }
}
